# Bert4Rec — эксперимент

Это реализация на PyTorch, не RePlay. Проверяем на том же разбиении, что и SasRec.

Результаты предыдущих запусков: 5 эпох — 0.034437, 20 эпох — 0.069536. В итоговую модель Bert4Rec не входит.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

if not Path('data/events.csv').exists() and Path('/content/rexis-main/data/events.csv').exists():
    os.chdir('/content/rexis-main')
assert Path('data/events.csv').exists(), 'Запусти блокнот из папки rexis-main'
from src.validation import leave_last_out
from src.metrics import recall_at_k_frame
from src.transformer_utils import fill_top100

SEED = 42
EPOCHS = 20
MAX_LEN = 50
HIDDEN = 64
HEADS = 2
LAYERS = 2
BATCH = 128

np.random.seed(SEED)
torch.manual_seed(SEED)
events = pd.read_csv('data/events.csv')
items = pd.read_csv('data/item_features.csv')
sample = pd.read_csv('data/submission_sample.csv')
train, truth = leave_last_out(events)
Path('outputs').mkdir(exist_ok=True)

## Модель и обучение

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset


class MaskedItemDataset(Dataset):
    def __init__(self, histories, count, max_len=50, seed=42):
        self.histories = histories
        self.max_len = max_len
        self.mask_id = count + 1
        self.seed = seed

    def __len__(self):
        return len(self.histories)

    def __getitem__(self, index):
        history = np.asarray(self.histories[index][-self.max_len:], dtype=np.int64) + 1
        x = np.zeros(self.max_len, dtype=np.int64)
        x[-len(history):] = history
        y = np.full(self.max_len, -100, dtype=np.int64)
        rng = np.random.default_rng(self.seed + index + int(torch.initial_seed() % 100000))
        positions = np.flatnonzero(x)
        masked = positions[rng.random(len(positions)) < 0.2]
        if len(masked) == 0:
            masked = rng.choice(positions, size=1)
        y[masked] = x[masked] - 1
        x[masked] = self.mask_id
        return torch.tensor(x), torch.tensor(y)


class Bert4Rec(nn.Module):
    def __init__(self, count, max_len=50, hidden=64, heads=2, layers=2):
        super().__init__()
        self.movies = nn.Embedding(count + 2, hidden, padding_idx=0)
        self.positions = nn.Embedding(max_len, hidden)
        layer = nn.TransformerEncoderLayer(hidden, heads, hidden * 4, 0.2, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, count)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        embedding = self.movies(x) + self.positions(positions)
        encoded = self.encoder(embedding, src_key_padding_mask=(x == 0))
        return self.head(self.norm(encoded))


@torch.no_grad()
def score_users(model, histories, count, device, batch_size=128, max_len=50):
    model.eval()
    result = []
    for start in range(0, len(histories), batch_size):
        batch = histories[start:start + batch_size]
        x = np.zeros((len(batch), max_len), dtype=np.int64)
        for i, history in enumerate(batch):
            history = np.asarray(history[-(max_len - 1):], dtype=np.int64) + 1
            x[i, -len(history) - 1:-1] = history
            x[i, -1] = count + 1
        scores = model(torch.from_numpy(x).to(device))[:, -1, :].cpu().numpy()
        result.append(scores)
    return np.concatenate(result)


In [ ]:
movie_ids = sorted(events.item_id.unique())
to_index = {item: i for i, item in enumerate(movie_ids)}
source = train.sort_values(['user_id', 'timestamp'], kind='mergesort')
groups = source.groupby('user_id').item_id.apply(list).to_dict()
users = sample.user_id.tolist()
histories = [[to_index[item] for item in groups[user]] for user in users]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

loader = DataLoader(
    MaskedItemDataset(histories, len(movie_ids), max_len=MAX_LEN, seed=SEED),
    batch_size=BATCH, shuffle=True, num_workers=0
)
net = Bert4Rec(len(movie_ids), max_len=MAX_LEN, hidden=HIDDEN, heads=HEADS, layers=LAYERS).to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.01)
for epoch in range(EPOCHS):
    net.train()
    losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = net(x)
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, len(movie_ids)), y.reshape(-1), ignore_index=-100
        )
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f'{epoch + 1}/{EPOCHS}: loss={np.mean(losses):.4f}')

## Результат на общей валидации

In [ ]:
score_matrix = score_users(net, histories, len(movie_ids), device, batch_size=BATCH, max_len=MAX_LEN)
seen = train.groupby('user_id').item_id.agg(set).to_dict()
rows = []
for j, user in enumerate(users):
    count = 0
    for idx in np.argsort(-score_matrix[j], kind='stable'):
        movie = movie_ids[int(idx)]
        if movie in seen.get(user, set()):
            continue
        rows.append((user, movie, float(score_matrix[j, idx])))
        count += 1
        if count == 100:
            break
recs = fill_top100(pd.DataFrame(rows, columns=['user_id', 'item_id', 'score']), train, users, items.item_id)
recs.to_csv('outputs/bert4rec_scores.csv', index=False)
recall = recall_at_k_frame(recs.groupby('user_id', sort=False).head(10), truth, 10)
print(f'Bert4Rec Recall@10: {recall:.6f}')